> Bad dtypes = slow code, wrong ML models, buggy API   
> Good dtypes = fast, predictable, production-ready systems

### Why dtype Matter
Every Pandas column has a dtype, which controls:
- Memory usage
- Speed of operations
- Valid values
- ML model behavior
- Serialization (JSON / DB)

### Inspecting dtypes
```python
df.dtypes
```

or more detail:

```python
Copy code
df.info()
```

Example output:
```go
user_id        int64
age          float64
city          object
last_login    datetime64[ns]
spend        float64
```

Red flags:
- `object` for strings
- `float64` where integers are expected
- dates stored as strings

### Numeric Types (int, float, nullable)
#### Standard integers & floats
```python
df["user_id"].dtype   # int64
df["salary"].dtype    # float64
```

### Nullable Numeric Types
| Type      | Use                         |
| --------- | --------------------------- |
| `Int64`   | Integer with missing values |
| `Float64` | Nullable float              |
| `boolean` | Nullable boolean            |

```python
s = pd.Series([1, None, 3], dtype="Int64")
```

- Keeps integers
- Supports missing values

### Type Conversion with `astype()`
```python
df["age"] = df["age"].astype("Int64")
```

### Safe Numeric Conversion (`to_numeric`)
```python
df["spend"] = pd.to_numeric(df["spend"], errors="coerce")
```

- Invalid values -> `NaN`
- Prevents crashes

### String Data: `object` vs `string`
```python
df["city"].dtype
```

`object`   
Means:
- Mixed types possible
- Slower
- Less predictable

#### Use Pandas `string` dtype
```python
df["city"] = df["city"].astype("string")
```

Benefits:
- Consistent missing values
- Faster string ops
- Cleaner semantics

### Categorical Data
Use when:
- Repeated values
- Finite set (city, status, role)

```python
df["city"] = df["city"].astype("category")
```

Why this matters:
- Lowers memory usage
- GroupBy faster
- Comparisons faster

> If a column has low cardinality -> use `category`

### Datetime Types
```python
df["joined_on"] = pd.to_datetime(df["joined_on"], errors="coerce")
```

Once converted, it gives:
```python
df["joined_on"].dt.year
df["joined_on"].dt.month
df["joined_on"].dt.day
```

### Boolean dtypes
```python
df["is_active"] = df["is_active"].astype("boolean")
```

Better than:
- `object`
- `int` (0/1)    

Supports missing values.

### Exercise 1
**Dataset:**
```python
df = pd.DataFrame({
    "user_id": [301, 302, 303, 304, 305, 306, 307],
    "age": [24, None, 29, 35, None, 41, 27],
    "gender": ["M", "F", None, "M", "F", None, "M"],
    "city": ["Pune", "Mumbai", "Delhi", None, "Pune", "Bangalore", None],
    "account_balance": [12000.50, 8500.00, None, 15000.75, None, 20000.00, 9800.25],
    "last_login": [
        "2024-02-01",
        None,
        "2024-02-05",
        "invalid_date",
        "2024-02-10",
        None,
        "2024-02-12"
    ],
    "is_premium_user": [True, None, False, True, None, True, False]
})
```

1. Print:
   - `df.info()`
   - `df.dtypes`


2. Answer:
   - Which columns have wrong or risky dtypes?
   - Which column is the most dangerous to leave unchanged?

In [32]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [301, 302, 303, 304, 305, 306, 307],
    "age": [24, None, 29, 35, None, 41, 27],
    "gender": ["M", "F", None, "M", "F", None, "M"],
    "city": ["Pune", "Mumbai", "Delhi", None, "Pune", "Bangalore", None],
    "account_balance": [12000.50, 8500.00, None, 15000.75, None, 20000.00, 9800.25],
    "last_login": [
        "2024-02-01",
        None,
        "2024-02-05",
        "invalid_date",
        "2024-02-10",
        None,
        "2024-02-12"
    ],
    "is_premium_user": [True, None, False, True, None, True, False]
})

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_id          7 non-null      int64  
 1   age              5 non-null      float64
 2   gender           5 non-null      object 
 3   city             5 non-null      object 
 4   account_balance  5 non-null      float64
 5   last_login       5 non-null      object 
 6   is_premium_user  5 non-null      object 
dtypes: float64(2), int64(1), object(4)
memory usage: 524.0+ bytes


In [34]:
df.dtypes

user_id              int64
age                float64
gender              object
city                object
account_balance    float64
last_login          object
is_premium_user     object
dtype: object

risky columns:
- `age` -> float because of missing values
- `last_login` -> object
- `is_premium_user` -> object

most dangerous:
- `last_login` 
   - Invalid strings don’t show as missing
   - Crashes on datetime ops
   - Breaks time-based ML features

### Exercise 2
1. Convert age to a nullable integer
2. Convert account_balance to a nullable float
3. Verify the dtypes

In [35]:
df["age"] = df["age"].astype("Int64")
df["account_balance"] = df["account_balance"].astype("Float64")
df.dtypes

user_id              int64
age                  Int64
gender              object
city                object
account_balance    Float64
last_login          object
is_premium_user     object
dtype: object

### Exercise 3
1. Convert `last_login` to datetime safely
2. Count how many `NaT` values exist

In [36]:
df["last_login"] = pd.to_datetime(df["last_login"], errors="coerce")

In [37]:
df["last_login"].isna().sum()

np.int64(3)

### Exercise 4
1. Convert:
   - `gender` -> `string`
   - `city` -> `category`
2. Explain why these two columns should NOT use the same dtype

In [38]:
df["gender"] = df["gender"].astype("string")
df["city"] = df["city"].astype("category")
df.dtypes

user_id                     int64
age                         Int64
gender             string[python]
city                     category
account_balance           Float64
last_login         datetime64[ns]
is_premium_user            object
dtype: object

`city` is a categorical feature

### Exercise 5
1. Convert `is_premium_user` to a **nullable boolean**
2. Explain why `int` (0/1) is a bad choice here

In [39]:
df["is_premium_user"] = df["is_premium_user"].astype("boolean")

This column has three possible states:
- `True`
- `False`
- **Unknown / Missing**

So it will give:
- `True` -> `1`
- `False` -> `0`
- `None` -> ERROR or silently coerced